In [ ]:
"""
Author: Sophie A. Liu
Date: 05/28/2026 10:58am
Purpose: isolating local expression activity around each immunofluorescent labeled cell
"""

In [1]:
# importing necessary libraries
import pandas as pd
import numpy as np
from tqdm import tqdm 
import os

In [2]:
# working directory
os.chdir("i:/Hu Lab/Sophie/1. Cell death/visium image manual spot selection/20260413_final_merge/data")

In [4]:
# V = pd.read_csv("0525_NMF_iso.csv")
V = pd.read_csv("0528_NMF_pd1.csv")

# S = pd.read_csv("iso7_coords_clean.csv") # spots from IF
S = pd.read_csv("pd1-9_coords_final.csv")  # spots from IF

In [38]:
# setting parameters/ initializing things
v_coords = V[["x", "y"]].to_numpy()
s_coords = S[["x", "y"]].dropna().to_numpy()

gene_cols = V.columns[3:25]

radius = 40            # balancing capturing enough cells but account for sparsity. 
                       # cell ~ 8 microns. study simplifies to 2D ignoring z-axis.

In [ ]:
print(gene_cols)

In [6]:
from scipy.spatial import cKDTree

In [7]:
def inputs(S, V, gene_cols, s_coords, v_coords):

    # KD-trees
    s_tree = cKDTree(s_coords)
    v_tree = cKDTree(v_coords)

    # encoding cell types as integers leads to faster processing
    type_map = {
        "tdtomato": 0,
        "gc3ai": 1,
        "cd8": 2,
        "lectin": 3
    }
    S_cells = np.array([type_map.get(x, -1) for x in S["cell_type"].values])

    # extracting gene matrix :)
    V_genes = V[gene_cols].to_numpy()

    return s_tree, v_tree, S_cells, V_genes

In [ ]:
# counts of each cell type in the neighborhood of a IF-labeled cell, as well as some derived metrics.
def counts_in_radius(center, s_tree, S_cells, radius):

    idx = s_tree.query_ball_point(center, r=radius)

    if len(idx) == 0:
        counts = np.zeros(4)   # for all four types, if nothing then set 0. Loops through all neighborhoods
    else:
        types = S_cells[idx]
        counts = np.bincount(types[types >= 0], minlength=4)        # our counts = [n_tdtomato, n_gc3ai, n_cd8, n_lectin]

    n_alive, n_dying, n_immune, n_endothelial = counts              # renaming the channels to what cell type they represent

    # calculating later metrics so I don't have to do it downstream
    tumor = n_alive + n_dying
    total = tumor + n_immune + n_endothelial
    prop_dying = n_dying / tumor                                    # will return some NaN
    exist_dying = 1 if prop_dying > 0 else 0                        # binarizing proportion dying
    eff_cont = prop_dying/ n_immune                                 # dying per cd8, ie. are some T-cells better at killing?
    eff_disc = n_dying / n_immune                   

    return counts, tumor, total, prop_dying, exist_dying, eff_cont, eff_disc

In [ ]:
# mean expression of each gene in the neighborhood of a IF-labeled cell, a better representation of gene expression than sums.
# spatial smoothing incorporated to account for sparsity effects
def get_gene_means(center, v_tree, V_genes, radius):
    idx = v_tree.query_ball_point(center, r=radius)

    if len(idx) == 0:
        return np.zeros(V_genes.shape[1])

    return V_genes[idx].mean(axis=0)

In [ ]:
def append_row(center, s_tree, v_tree, S_types, V_genes, radius):

    counts, tumor, total, prop_dying, exist_dying, eff_cont, eff_disc = counts_in_radius(
        center, s_tree, S_types, radius
    )

    gene_means = get_gene_means(
        center, v_tree, V_genes, radius
    )

    row = np.concatenate([
        np.array([center[0], center[1]]),
        counts,
        np.array([tumor, total, prop_dying, exist_dying, eff_cont, eff_disc]),
        gene_means
    ])

    return row

In [ ]:
# function for final assembly/joining of neighborhoodresults. didn't vectorize so slower but not as intuitive for me. 
def compute_neighborhoods(
    S, V, s_coords, v_coords, gene_cols, radius):

    s_tree, v_tree, S_types, V_genes = inputs(
        S, V, gene_cols, s_coords, v_coords
    )

    n_centers = len(s_coords)
    n_genes = V_genes.shape[1]

    results = np.zeros((n_centers, 11 + n_genes))

    for i, center in enumerate(tqdm(s_coords, desc="Processing")):
        results[i] = append_row(
            center, s_tree, v_tree, S_types, V_genes, radius
        )

    columns = (
        ["cx", "cy",
         "n_alive", "n_dying", "n_immune", "n_lectin",
         "tumor", "all", "prop_dying", "exist_dying", "eff_cont", "eff_disc"]
        + list(gene_cols)
    )

    return pd.DataFrame(results, columns=columns)

In [ ]:
# after the progress bar ends, it may still take a while. just a heads up, sorry
df = compute_neighborhoods(
    S=S,
    V=V,
    s_coords=s_coords,
    v_coords=v_coords,
    gene_cols=gene_cols,
    radius=radius
)

df = df.join(S[["cell_type", "sample"]])         # maintaining cell type and sample info for downstream

In [40]:
df_clean = df.dropna()                           # removing for NaN efficacy denominator

In [20]:
df_eff = df_clean[df_clean["n_immune"] != 0.0]   # optional and tbh cleans too much

In [ ]:
# helps restore independence (not completely) by sampling non-overlapping neighborhoods using a greedy algorithm.
def non_overlapping(df, n, radius, seed):
    rng = np.random.default_rng(seed)

    coords = df[['cx', 'cy']].to_numpy()
    remaining_idx = np.arange(len(coords))

    selected_idx = []

    while len(selected_idx) < n and len(remaining_idx) > 0:
        pick_i = rng.choice(remaining_idx)
        selected_idx.append(pick_i)

        # tree around above point
        tree = cKDTree(coords[remaining_idx])

        neighbors = tree.query_ball_point(coords[pick_i], r=radius)

        to_remove = set(remaining_idx[neighbors])

        # keeping only points not removed
        remaining_idx = np.array([i for i in remaining_idx if i not in to_remove])

    return df.iloc[selected_idx].copy()

In [41]:
df_sub = non_overlapping(df_clean, n = 1000,                        # comparability with pd1. still >383 so we good
                                       radius=radius*2, seed=42)    # the answer to the ultimate question of life, the universe, and everything.

In [ ]:
# verify output
print(df_sub.iloc[300:305, ])
df_sub.shape

In [43]:
df_sub.to_csv("0602_22NMF_pd1_80incl.csv", index=False)